In [ ]:
import requests
from datetime import datetime
import json
import pyodbc
import pandas as pd
import time
from azure.storage.blob import BlobServiceClient
import os

In [2]:
current_time=datetime.now()
date_year=current_time.year
date_month=current_time.month
date_date=current_time.date()

In [3]:
# read file
with open("config.json", "r") as file:
    config = json.load(file)

In [4]:
# # extract values

username = config["username"]
password = config["password"]
database = config["database_name"]
server = config["server_name"]

In [5]:
# Connection String

conn_str = (
    f"Driver={{ODBC Driver 17 for SQL Server}};"
    f"Server={server};"
    f"Database={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;TrustServerCertificate=no;"
)

In [6]:
# conn = pyodbc.connect(conn_str)
# query = f"SELECT CloseOfBizDate FROM [cim-prod-d2-bc].[Config].[t_Processconfig] WHERE BrandKey='ARCDK'"
# data = pd.read_sql(query, conn)



In [7]:
## Getting access token

def get_access_token(tenant_id, client_id, client_secret, max_retries=3):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "https://api.businesscentral.dynamics.com/.default"
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, data=payload, timeout=10)

            # ✅ Success
            if response.status_code == 200:
                return response.json()["access_token"]

            # 🔐 Auth errors (DON'T retry blindly)
            elif response.status_code in [400, 401]:
                raise Exception(f"Auth Error: {response.text}")

            # 🔁 Retry for server issues
            elif response.status_code in [429, 500, 502, 503]:
                print(f"Retrying... Attempt {attempt+1}")
                time.sleep(2 ** attempt)  # exponential backoff

            else:
                raise Exception(f"Unexpected Error: {response.text}")

        except requests.exceptions.Timeout:
            print(f"Timeout... retrying {attempt+1}")
            time.sleep(2 ** attempt)

        except requests.exceptions.RequestException as e:
            print(f"Network Error: {str(e)}")
            time.sleep(2 ** attempt)

    raise Exception("Failed to get access token after retries")


In [8]:
tenant_id=config["tenant_id"]
client_id=config["client_id"]
client_secret=config["client_secret"]
tables = config["tables"]

In [9]:
access_token=get_access_token(tenant_id, client_id, client_secret)   

In [10]:

### Fetching data 

def fetch_data(base_url, tenant_id, company_id, table_name, access_token):
    
    if table_name =="salesInvoiceLines":
        url = f"{base_url}/v2.0/{tenant_id}/production/api/v2.0/companies({company_id})/salesInvoices?&$expand=salesInvoiceLines"
    elif table_name =="salesCreditMemoLines":
        url = f"{base_url}/v2.0/{tenant_id}/production/api/v2.0/companies({company_id})/salesCreditMemos?&$expand=salesCreditMemoLines"
    elif table_name =="CustomerCard":
        url = f"{base_url}/v2.0/{tenant_id}/production/ODataV4/Company('Arctiko')/PBI_Customers"
    elif table_name =="Salespersons":
        url = f"{base_url}/v2.0/{tenant_id}/production/ODataV4/Company('Arctiko')/Salespersons"
    else:
        url = f"{base_url}/v2.0/{tenant_id}/production/api/v2.0/companies({company_id})/{table_name}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers)
    all_lines=[]
    if response.status_code == 200:
        records= response.json().get('value', [])
        if table_name =="salesInvoiceLines":
            for record in records:
                # Get the expanded lines directly from the shipment object
                lines = record.get('salesInvoiceLines', [])
                all_lines.extend(lines)
            return pd.DataFrame(all_lines)
        if table_name =="salesCreditMemoLines":
            for record in records:
                # Get the expanded lines directly from the shipment object
                lines = record.get('salesCreditMemoLines', [])
                all_lines.extend(lines)
            return pd.DataFrame(all_lines)
        else:
            return pd.DataFrame(records)

    else:
        raise Exception(f"API Error: {response.status_code} - {response.text}")


In [11]:
base_url ='https://api.businesscentral.dynamics.com'
company_id = config["company_id"]

In [12]:
# =========================
# SAVE FILE
# =========================

os.makedirs(
    f"Files/{date_year}/{date_month}/{date_date}",
    exist_ok=True
)

def save_to_json(df, table_name):

    file_name = (
        f"Files/{date_year}/{date_month}/{date_date}/"
        f"{table_name}.json"
    )

    df.to_json(
        file_name,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"Saved: {file_name} 🚀")

In [13]:
def clean_dataframe(df):

    # Remove metadata columns
    df = df.loc[:, ~df.columns.str.startswith("@")]

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Strip column names
    df.columns = [col.strip() for col in df.columns]

    return df

In [ ]:
# =========================
# MAIN EXECUTION
# =========================

for table in tables:

    try:

        print(f"Fetching: {table}")

        raw_data = fetch_data(
            base_url,
            tenant_id,
            company_id,
            table,
            access_token
        )

        if not raw_data:
            print(f"No data found for {table}")

        # Save
        df = raw_data.drop(columns=["@odata.etag"], errors="ignore")
        save_to_json(df, table)

    except Exception as e:

        print(f"Failed for {table}: {str(e)}")

In [15]:
conn_str = config["conn_str"]
container_name = "bc-to-end-azure-lakehouse"

In [16]:
def upload_to_blob(file_path, blob_name, container_name, conn_str):
    
    blob_service_client = BlobServiceClient.from_connection_string(conn_str)
    
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    with open(file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

    print(f"Uploaded: {blob_name} 🚀")

In [ ]:
import os

local_folder = f"Files/{date_year}/{date_month}/{date_date}/"

blob_base_path = f"bronze/{date_year}/{date_month}/{date_date}"

for root, dirs, files in os.walk(local_folder):

    for file in files:

        # Full local file path
        local_file_path = os.path.join(root, file)
        # Blob path
        blob_path = f"{blob_base_path}"

        print("Local File:", local_file_path)
        print("Blob Path :", blob_path)
        upload_to_blob(local_file_path,f"{blob_base_path}/{file}",container_name,conn_str)
        print("-" * 50)